In [1291]:
from pathlib import Path
import json
import re
from datetime import datetime

import spacy

In [1292]:
SILVER_FOLDER = Path("../../Data/silver")
GOLD_META_FOLDER = Path("../../Data/gold_meta")

GOLD_META_FOLDER.mkdir(parents=True, exist_ok=True)

In [1293]:
SPACY_MODELS = {
    "en": "en_core_web_sm",
    "nl": "nl_core_news_sm"
}

In [1294]:
def clean_person_candidate_line(line):
    line = clean_entity_text(line)

    stop_patterns = [
        r"\buitgave\b.*$",
        r"\bedition\b.*$",
        r"\bissue\b.*$",
        r"\bversion\b.*$",
        r"\bversie\b.*$",
        r"\bpublication\b.*$",
        r"\bpublicatie\b.*$"
    ]

    for pattern in stop_patterns:
        line = re.sub(pattern, "", line, flags=re.IGNORECASE).strip()

    return clean_entity_text(line)

In [1295]:
def get_lines(text, max_lines=None):
    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    return lines[:max_lines] if max_lines else lines

In [1296]:
def looks_like_section_or_noise(line):
    lower = line.lower().strip(" :-")

    noise_terms = [
        "inhoudsopgave", "contents", "table of contents",
        "inleiding", "introduction",
        "colofon", "colophon", "credits",
        "thema", "themes",
        "pagina", "page",
        "conclusies", "conclusions"
    ]

    if lower in noise_terms:
        return True

    if re.match(r"^\d+$", line):
        return True

    if re.match(r"^\d+\.?\s+", line):
        return True

    if lower.startswith(("grafiek", "tabel", "figuur", "figure", "table")):
        return True

    return False

In [1297]:
def looks_like_date_line(line):
    month_year_pattern = (
        r"\b(?:januari|februari|maart|april|mei|juni|juli|augustus|"
        r"september|oktober|november|december|January|February|March|"
        r"April|May|June|July|August|September|October|November|December)"
        r"\s+\d{4}\b"
    )

    return bool(re.search(month_year_pattern, line, flags=re.IGNORECASE))

In [1298]:
def looks_like_subtitle(line):
    lower = line.lower()

    subtitle_starts = [
        "onderzoek naar",
        "research into",
        "study of",
        "rapportage over",
        "report on",
        "analyse van",
        "analysis of",
        "factsheet over",
        "samenvatting van"
    ]

    return any(lower.startswith(term) for term in subtitle_starts)

In [1299]:
def extract_metadata_sections(text, max_lines_per_section=40):
    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    section_headers = [
        "colofon",
        "colophon",
        "imprint",
        "credits",
        "publication details",
        "document details",
        "about this publication",
        "over deze publicatie"
    ]

    sections = []

    for index, line in enumerate(lines):
        lower = line.lower().strip(" :-")

        if lower in section_headers:
            section_lines = lines[index:index + max_lines_per_section]
            sections.append("\n".join(section_lines))

    return sections

In [1300]:
def looks_like_metadata_label(text):
    labels = [
        "author", "authors", "auteur", "auteurs",
        "contributor", "contributors", "bijdrager", "bijdragers",
        "editor", "editors", "redactie", "eindredactie",
        "compiled by", "compiler", "samenstelling",
        "prepared by", "written by", "created by", "developed by",
        "uitgevoerd door", "opgesteld door",
        "design", "ontwerp", "druk", "publisher", "uitgever"
    ]

    text = text.lower().strip(" :-")

    return text in labels

In [1301]:
def extract_label_value_pairs_from_section(section):
    lines = [
        line.strip()
        for line in section.splitlines()
        if line.strip()
    ]

    pairs = []

    for index, line in enumerate(lines):
        clean_label = line.lower().strip(" :-")

        # Case 1: label and value on same line
        if ":" in line:
            label, value = line.split(":", 1)
            pairs.append((label.strip(), value.strip()))
            continue

        # Case 2: label on one line, value on next line
        if index + 1 < len(lines):
            next_line = lines[index + 1].strip()

            if looks_like_metadata_label(clean_label) and not looks_like_metadata_label(next_line.lower()):
                pairs.append((line.strip(), next_line))

    return pairs

In [1302]:
def looks_like_document_title_or_section(line):
    lower = line.lower().strip(" :-")

    document_terms = [
        "risk", "assessment", "impact", "summary", "sector",
        "industry", "climate", "change", "report", "rapport",
        "deel", "part", "chapter", "section", "hoofdstuk",
        "key characteristics", "inleiding", "introduction"
    ]

    if any(term in lower for term in document_terms):
        return True

    if re.match(r"^[A-Z]\.\s+", line):
        return True

    if re.match(r"^[A-Z]\.\d+", line):
        return True

    return False

In [1303]:
def merge_multiline_title(lines, start_index, max_lines=5):
    title_lines = [lines[start_index]]

    stop_prefixes = [
        "part ",
        "section ",
        "chapter ",
        "volume ",
        "deel ",
        "hoofdstuk "
    ]

    for line in lines[start_index + 1:start_index + max_lines]:
        lower = line.lower().strip()

        if any(lower.startswith(prefix) for prefix in stop_prefixes):
            break

        if looks_like_date_line(line):
            break

        if looks_like_contact_or_address_line(line):
            break

        if looks_like_section_or_noise(line):
            break

        if len(line.split()) > 8:
            break

        if line.endswith("."):
            break

        title_lines.append(line)

    return re.sub(r"\s+", " ", " ".join(title_lines)).strip()

In [1304]:
def looks_like_contact_or_address_line(line):
    lower = line.lower().strip()

    # Postcode patterns
    if re.search(r"\b\d{4}\s?[a-z]{2}\b", lower):
        return True

    # Email / URL
    if re.search(r"@[a-z0-9.-]+\.[a-z]{2,}", lower):
        return True

    if "www." in lower:
        return True

    # Explicit contact words
    if re.search(
        r"\b(postbus|postcode|telefoon|tel\.?|email|e-mail|fax|mobile|phone|contact)\b",
        lower
    ):
        return True

    # Phone-like numbers
    if re.search(r"\+?\d[\d\s().-]{7,}\d", line):
        return True

    # Address-like pattern:
    # "Some Street 12"
    # "Mainroad 5A"
    # "Broadway 221B"
    if re.search(
        r"^[A-ZÀ-Ý][A-Za-zÀ-ÿ'’\-]+\s(?:[A-ZÀ-Ý][A-Za-zÀ-ÿ'’\-]+\s){0,4}\d+[A-Za-z]?$",
        line
    ):
        return True

    # Very short lines with many digits
    digit_ratio = sum(c.isdigit() for c in line) / max(1, len(line))

    if digit_ratio > 0.20 and len(line.split()) <= 6:
        return True

    return False

In [1305]:
def load_spacy_model(language):
    model_name = SPACY_MODELS.get(language)

    if not model_name:
        return None

    try:
        return spacy.load(model_name)
    except OSError:
        print(f"spaCy model not installed: {model_name}")
        return None

In [1306]:
def read_json(path):
    if not path.exists():
        return {}

    return json.loads(path.read_text(encoding="utf-8"))

In [1307]:
def get_document_language(document_id):
    silver_json = SILVER_FOLDER / f"{document_id}_silver.json"
    silver_data = read_json(silver_json)

    return silver_data.get("language", "en")

In [1308]:
def read_metadata_text(document_id):
    silver_json = SILVER_FOLDER / f"{document_id}_silver.json"

    if not silver_json.exists():
        return ""

    silver_data = json.loads(silver_json.read_text(encoding="utf-8"))
    metadata_text_file = silver_data.get("metadata_text_file")

    if not metadata_text_file:
        return ""

    path = Path(metadata_text_file)

    if not path.exists():
        return ""

    return path.read_text(encoding="utf-8")

In [1309]:
def normalize_text(text):
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = text.replace("\u00a0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

In [1310]:
def get_front_matter(text, max_lines=120):
    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    return "\n".join(lines[:max_lines])

In [1311]:
def extract_contributors(text, doc=None):
    candidates = extract_contributor_candidates(text)

    best_by_name = {}

    for candidate in candidates:
        key = candidate["text"].lower()

        if key not in best_by_name or candidate["score"] > best_by_name[key]["score"]:
            best_by_name[key] = candidate

    selected = sorted(
        best_by_name.values(),
        key=lambda x: x["score"],
        reverse=True
    )

    high_or_medium = [
        item for item in selected
        if item["score"] >= 65
    ]

    structured = {
        "persons": [],
        "organizations": [],
        "authors": [],
        "editors": [],
        "compiled_by": [],
        "prepared_by": [],
        "commissioned_by": [],
        "confidence": "low"
    }

    for item in high_or_medium:
        text = item["text"]
        role = item["role"]

        structured["persons"].append(text)

        if role in structured:
            structured[role].append(text)

    for key in structured:
        if isinstance(structured[key], list):
            structured[key] = unique_keep_order(structured[key])

    if any(item["score"] >= 90 for item in high_or_medium):
        structured["confidence"] = "high"
    elif high_or_medium:
        structured["confidence"] = "medium"

    return structured

In [1312]:
def extract_subtitle(text, title=None):
    lines = get_lines(text, max_lines=80)

    if not title:
        return None

    title_words = title.split()
    title_end_index = None

    for index in range(len(lines)):
        window = " ".join(lines[index:index + 5])
        window = re.sub(r"\s+", " ", window).strip()

        if window.startswith(title):
            used_words = 0

            for offset, line in enumerate(lines[index:index + 5]):
                used_words += len(line.split())

                if used_words >= len(title_words):
                    title_end_index = index + offset
                    break

            break

    if title_end_index is None:
        return None

    subtitle_lines = []

    for line in lines[title_end_index + 1:title_end_index + 6]:
        lower = line.lower().strip()

        if looks_like_date_line(line):
            break

        if looks_like_contact_or_address_line(line):
            break

        if is_valid_person(clean_person_candidate_line(line)):
            break

        if len(line.split()) > 10:
            break

        subtitle_lines.append(line)

    subtitle = " ".join(subtitle_lines)
    subtitle = re.sub(r"\s+", " ", subtitle).strip(" :-")

    return subtitle if subtitle else None

In [1313]:
def extract_title(text):
    lines = get_lines(text, max_lines=50)
    candidates = []

    for index, line in enumerate(lines):
        words = line.split()

        if looks_like_contact_or_address_line(line):
            continue

        if looks_like_section_or_noise(line):
            continue

        if looks_like_date_line(line):
            continue

        if len(words) < 2 or len(words) > 18:
            continue

        digit_ratio = sum(c.isdigit() for c in line) / max(1, len(line))

        if digit_ratio > 0.35:
            continue

        score = 0

        # Early lines are usually better
        score += max(0, 50 - index)

        # Title-like length
        if 2 <= len(words) <= 8:
            score += 20

        if 9 <= len(words) <= 14:
            score += 8

        # Often title contains year/range
        if re.search(r"\b\d{4}([-/]\d{2,4})?\b", line):
            score += 25

        # All caps is often a title in factsheets
        alpha_chars = [c for c in line if c.isalpha()]
        if alpha_chars:
            upper_ratio = sum(c.isupper() for c in alpha_chars) / len(alpha_chars)
            if upper_ratio > 0.65:
                score += 10

        # Subtitle penalty
        if looks_like_subtitle(line):
            score -= 35

        # Avoid sentence-like title
        if line.endswith("."):
            score -= 20

        merged_line = merge_multiline_title(lines, index)

        if merged_line != line:
            score += 30

        candidates.append((score, index, merged_line))

    if not candidates:
        return lines[0] if lines else None

    best = sorted(candidates, key=lambda x: x[0], reverse=True)[0]
    return best[2]

In [1314]:
def classify_contributor_label(label):
    label = label.lower().strip(" :-")

    mapping = {
        "author": "authors",
        "authors": "authors",
        "auteur": "authors",
        "auteurs": "authors",
        "editor": "editors",
        "editors": "editors",
        "redactie": "editors",
        "eindredactie": "editors",
        "compiled by": "compiled_by",
        "compiler": "compiled_by",
        "samenstelling": "compiled_by",
        "prepared by": "prepared_by",
        "written by": "authors",
        "created by": "prepared_by",
        "developed by": "prepared_by",
        "uitgevoerd door": "prepared_by",
        "opgesteld door": "prepared_by",
        "commissioned by": "commissioned_by",
        "in opdracht van": "commissioned_by",
        "opdrachtgever": "commissioned_by",
        "colofon": "persons",
        "colophon": "persons",
        "credits": "persons"
    }

    return mapping.get(label, "persons")

In [1315]:
def add_contributor_candidate(candidates, name, role, evidence, score):
    name = clean_entity_text(name)

    if not is_valid_person(name):
        return

    candidates.append({
        "text": name,
        "type": "person",
        "role": role,
        "evidence": evidence,
        "score": score
    })

In [1316]:
def extract_contributor_candidates(text):
    lines = get_lines(text, max_lines=600)

    contributor_labels = [
        "author", "authors", "auteur", "auteurs",
        "contributor", "contributors", "bijdrager", "bijdragers",
        "editor", "editors", "redactie", "eindredactie",
        "compiled by", "compiler", "samenstelling",
        "prepared by", "written by", "created by", "developed by",
        "uitgevoerd door", "opgesteld door",
        "commissioned by", "in opdracht van", "opdrachtgever",
        "colofon", "colophon", "credits"
    ]

    stop_labels = [
        "design", "ontwerp", "druk", "publisher", "uitgever",
        "contact", "email", "e-mail", "telefoon", "www",
        "thema", "themes", "inhoud", "contents",
        "meer weten", "more information",
        "inleiding", "introduction"
    ]

    candidates = []

    # A. Explicit label line followed by one or more names
    for index, line in enumerate(lines):
        label = line.lower().strip(" :-")

        if label not in contributor_labels:
            continue

        role = classify_contributor_label(label)

        for lookahead in range(1, 12):
            if index + lookahead >= len(lines):
                break

            candidate = clean_person_candidate_line(lines[index + lookahead])
            candidate_lower = candidate.lower().strip(" :-")

            if candidate_lower in contributor_labels:
                continue

            if any(candidate_lower.startswith(stop) for stop in stop_labels):
                break

            if looks_like_contact_or_address_line(candidate):
                continue

            if looks_like_metadata_label(candidate):
                break

            if is_valid_person(candidate):
                add_contributor_candidate(
                    candidates,
                    candidate,
                    role,
                    "explicit_label_following_lines",
                    95
                )
                continue

            if len(candidate.split()) > 6:
                break

    # B. Label and value on same line
    label_pattern = re.compile(
        r"^(author|authors|auteur|auteurs|contributor|contributors|bijdrager|bijdragers|"
        r"editor|editors|redactie|eindredactie|compiled by|compiler|samenstelling|"
        r"prepared by|written by|created by|developed by|uitgevoerd door|opgesteld door|"
        r"commissioned by|in opdracht van|opdrachtgever)"
        r"\s*[:\-]?\s+(.+)$",
        flags=re.IGNORECASE
    )

    for line in lines:
        match = label_pattern.match(line)

        if not match:
            continue

        role = classify_contributor_label(match.group(1))
        value = clean_entity_text(match.group(2))

        parts = re.split(r",|;|\band\b|\ben\b|&|\|", value)

        for part in parts:
            add_contributor_candidate(
                candidates,
                part,
                role,
                "explicit_label_same_line",
                90
            )

    # C. Cover-page names after date
    for index, line in enumerate(lines[:40]):
        if not looks_like_date_line(line):
            continue

        for lookahead in range(1, 8):
            if index + lookahead >= len(lines):
                break

            candidate = clean_person_candidate_line(lines[index + lookahead])

            if looks_like_section_or_noise(candidate):
                break

            if looks_like_contact_or_address_line(candidate):
                continue

            if len(candidate.split()) > 6:
                break

            if is_valid_person(candidate):
                add_contributor_candidate(
                    candidates,
                    candidate,
                    "authors",
                    "cover_page_after_date",
                    80
                )

    # D. Cover-page names directly after title/subtitle block
    for index, line in enumerate(lines[:25]):
        if looks_like_date_line(line):
            continue

        candidate = clean_person_candidate_line(line)

        if is_valid_person(candidate):
            add_contributor_candidate(
                candidates,
                candidate,
                "authors",
                "cover_page_name",
                65
            )

    return candidates

In [1317]:
def extract_description(text, title=None, max_words=90):
    bad_description_starts = [
        "colofon",
        "colophon",
        "credits",
        "thema",
        "themes",
        "inhoud",
        "contents"
    ]

    intro_terms = [
        "inleiding",
        "introduction",
        "dit rapport",
        "this report",
        "deze publicatie",
        "this publication",
        "dit themarapport",
        "this document"
    ]

    paragraphs = [
        p.strip()
        for p in re.split(r"\n\s*\n", text)
        if p.strip()
    ]

    scored = []

    for index, paragraph in enumerate(paragraphs[:50]):
        paragraph = re.sub(r"\s+", " ", paragraph)
        paragraph_lower = paragraph.lower()
        words = paragraph.split()

        if any(paragraph_lower.startswith(term) for term in bad_description_starts):
            continue

        if len(words) < 35:
            continue

        if title and title.lower() in paragraph_lower:
            continue

        if looks_like_table_of_contents(paragraph):
            continue

        digit_ratio = sum(c.isdigit() for c in paragraph) / max(1, len(paragraph))
        punctuation_ratio = sum(not c.isalnum() and not c.isspace() for c in paragraph) / max(1, len(paragraph))

        if digit_ratio > 0.10:
            continue

        if punctuation_ratio > 0.18:
            continue

        score = 0
        score += max(0, 50 - index)

        if any(term in paragraph_lower for term in intro_terms):
            score += 25

        if paragraph.endswith("."):
            score += 10

        scored.append((score, paragraph))

    if scored:
        best = sorted(scored, key=lambda x: x[0], reverse=True)[0][1]
        return " ".join(best.split()[:max_words])

    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    blocks = []
    current = []
    skipping_metadata_block = False

    for line in lines:
        line_lower = line.lower()

        if any(line_lower.startswith(term) for term in bad_description_starts):
            current = []
            skipping_metadata_block = True
            continue

        if skipping_metadata_block:
            if re.match(r"^\d+(\.\d+)*\.?\s+", line):
                skipping_metadata_block = False
            else:
                continue

        if title and title.lower() in line_lower:
            continue

        if looks_like_contact_or_address_line(line):
            continue

        if looks_like_table_of_contents(line):
            continue

        if looks_like_metadata_label(line):
            continue

        if re.match(r"^\d+(\.\d+)*\.?\s+", line):
            continue

        if len(line.split()) < 5:
            continue

        current.append(line)

        if line.endswith(".") and len(" ".join(current).split()) >= 35:
            blocks.append(" ".join(current))
            current = []

    for block in blocks:
        block = re.sub(r"\s+", " ", block)
        words = block.split()

        digit_ratio = sum(c.isdigit() for c in block) / max(1, len(block))
        punctuation_ratio = sum(not c.isalnum() and not c.isspace() for c in block) / max(1, len(block))

        if digit_ratio <= 0.10 and punctuation_ratio <= 0.18:
            return " ".join(words[:max_words])

    return None

In [1318]:
def extract_dates(text, doc=None):
    front = get_front_matter(text, max_lines=60)

    month_year_pattern = (
        r"\b(?:januari|februari|maart|april|mei|juni|juli|augustus|"
        r"september|oktober|november|december|January|February|March|"
        r"April|May|June|July|August|September|October|November|December)"
        r"\s+\d{4}\b"
    )

    full_date_pattern = r"\b\d{1,2}[-/]\d{1,2}[-/]\d{4}\b"

    dates = []
    dates.extend(re.findall(month_year_pattern, front, flags=re.IGNORECASE))
    dates.extend(re.findall(full_date_pattern, front, flags=re.IGNORECASE))

    dates = unique_keep_order(dates)

    start_date, end_date = extract_date_range(front)

    return {
        "dates_found": dates[:10],
        "publication_date": dates[0] if dates else None,
        "start_date": start_date,
        "end_date": end_date
    }

In [1319]:
def extract_date_range(text):
    front = get_front_matter(text)

    patterns = [
        r"\b(?:from|van)\s+(.{3,30}?\d{4})\s+(?:to|tot|until|t/m)\s+(.{3,30}?\d{4})\b",
        r"\b(?:start date|startdatum)[:\s]+(.{3,30}?\d{4}).{0,40}(?:end date|einddatum)[:\s]+(.{3,30}?\d{4})",
        r"\b(\d{1,2}[-/]\d{1,2}[-/]\d{4})\s*[-–]\s*(\d{1,2}[-/]\d{1,2}[-/]\d{4})\b"
    ]

    for pattern in patterns:
        match = re.search(pattern, front, flags=re.IGNORECASE | re.DOTALL)
        if match:
            return match.group(1).strip(), match.group(2).strip()

    return None, None

In [1320]:
def is_valid_contributor(name):
    if not name:
        return False

    words = name.split()

    if len(words) < 1 or len(words) > 6:
        return False

    if len(name) < 3 or len(name) > 70:
        return False

    if not any(char.isalpha() for char in name):
        return False

    lower = name.lower()

    bad_terms = [
        "will",
        "would",
        "should",
        "could",
        "project",
        "direction",
        "assessment",
        "recommendations",
        "chapter",
        "section",
        "appendix",
        "table",
        "figure",
        "page",
        "draft",
        "http",
        "www",
        "pdf",
        "available",
        "accessed"
    ]

    if any(term in lower for term in bad_terms):
        return False

    return True

In [1321]:
def clean_entity_text(text):
    text = re.sub(r"\s+", " ", text)
    text = text.strip(" .,:;()[]{}")

    return text

In [1322]:
def is_valid_organization(text):
    if not text:
        return False

    if len(text) < 3 or len(text) > 100:
        return False

    if len(text.split()) > 10:
        return False

    lower = text.lower()

    bad_terms = [
        "draft",
        "table",
        "figure",
        "appendix",
        "page",
        "chapter",
        "section",
        "recommendations",
        "summary",
        "http",
        "www",
        "pdf",
        "available",
        "accessed",
        "introduction"
    ]

    if any(term in lower for term in bad_terms):
        return False

    if not any(c.isalpha() for c in text):
        return False

    return True

In [1323]:
def is_valid_person(text):
    if not text:
        return False

    words = text.split()

    if len(words) < 2 or len(words) > 5:
        return False

    if len(text) < 5 or len(text) > 70:
        return False

    lower = text.lower()

    if looks_like_document_title_or_section(text):
        return False
    
    bad_terms = [
        "as they",
        "they would",
        "contact person",
        "for more",
        "project",
        "draft",
        "appendix",
        "table",
        "figure",
        "telefoon",
        "postbus",
        "www",
        "info@"
    ]

    if any(term in lower for term in bad_terms):
        return False

    lowercase_name_particles = {
        "van", "de", "der", "den", "ten", "ter", "op", "aan",
        "von", "zu", "da", "di", "del", "la", "le"
    }

    meaningful_words = [
        word for word in words
        if word.lower() not in lowercase_name_particles
    ]

    if len(meaningful_words) < 2:
        return False

    if not all(word[:1].isupper() for word in meaningful_words):
        return False

    return True

In [1324]:
def extract_contact_person(text, doc=None):
    front = get_front_matter(text, max_lines=80)

    email = extract_email(front)
    phone = extract_phone(front)

    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    label_pattern = re.compile(
        r"^(contact person|contactpersoon|contact|for more information|meer informatie|email|e-mail)\s*[:\-]\s*(.+)$",
        flags=re.IGNORECASE
    )

    for line in lines[:200]:
        match = label_pattern.match(line)

        if not match:
            continue

        value = match.group(2).strip()

        labeled_email = extract_email(value)
        labeled_phone = extract_phone(value)

        name = None

        name_match = re.search(
            r"\b[A-ZÀ-Ý][a-zà-ÿ]+(?:\s+(?:van|de|der|den|ten|ter|op|aan|von|zu|da|di|del|la|le))?(?:\s+[A-ZÀ-Ý][a-zà-ÿ]+){1,3}\b",
            value
        )

        if name_match:
            candidate = name_match.group(0)
            if is_valid_person(candidate):
                name = candidate

        return {
            "name": name,
            "email": labeled_email or email,
            "phone": labeled_phone or phone,
            "confidence": "high"
        }

    return {
        "name": None,
        "email": email,
        "phone": phone,
        "confidence": "medium" if email or phone else "low"
    }

In [1325]:
def extract_email(text):
    match = re.search(
        r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}",
        text
    )

    return match.group(0) if match else None

In [1326]:
def extract_labeled_contributors(text):
    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    contributor_labels = [
        "author", "authors", "auteur", "auteurs",
        "contributor", "contributors", "bijdrager", "bijdragers",
        "editor", "editors", "redactie", "eindredactie",
        "compiled by", "compiler", "samenstelling",
        "prepared by", "written by", "created by", "developed by",
        "uitgevoerd door", "opgesteld door",
        "colofon", "colophon", "credits"
    ]

    stop_labels = [
        "design", "ontwerp", "druk", "publisher", "uitgever",
        "contact", "email", "e-mail", "telefoon", "www",
        "thema", "themes", "inhoud", "contents",
        "meer weten", "more information"
    ]

    contributors = []

    for index, line in enumerate(lines[:500]):
        label = line.lower().strip(" :-")

        if label not in contributor_labels:
            continue

        for lookahead in range(1, 12):
            if index + lookahead >= len(lines):
                break

            candidate = clean_entity_text(lines[index + lookahead])
            candidate_lower = candidate.lower().strip(" :-")

            if candidate_lower in contributor_labels:
                continue

            if any(candidate_lower.startswith(stop) for stop in stop_labels):
                break

            if looks_like_contact_or_address_line(candidate):
                continue

            if looks_like_metadata_label(candidate):
                break

            if is_valid_person(candidate):
                contributors.append(candidate)
                continue

            # Stop once real prose starts
            if len(candidate.split()) > 6:
                break

    return unique_keep_order(contributors)

In [1327]:
def extract_phone(text):
    phone_patterns = [
        r"(?:telefoon|tel\.?|t)\s*[:\-]?\s*(\(?0\d{2,4}\)?[\s.-]?\d{3,4}[\s.-]?\d{3,4})",
        r"\b(\(?0\d{2,4}\)?[\s.-]?\d{3,4}[\s.-]?\d{3,4})\b"
    ]

    for pattern in phone_patterns:
        match = re.search(pattern, text, flags=re.IGNORECASE)

        if match:
            phone = match.group(1).strip()

            # reject postcode-like values
            if re.fullmatch(r"\d{4}\s?[A-Z]{2}", phone, flags=re.IGNORECASE):
                continue

            return phone

    return None

In [1328]:
def looks_like_table_of_contents(text):
    lines = [line.strip() for line in text.splitlines() if line.strip()]

    if len(lines) < 5:
        return False

    numbered_lines = sum(
        bool(re.match(r"^\d+(\.\d+)*\.?\s+", line))
        for line in lines
    )

    dotted_lines = sum("." * 4 in line for line in lines)

    short_lines = sum(len(line.split()) <= 8 for line in lines)

    return (
        numbered_lines >= 3
        or dotted_lines >= 3
        or short_lines / len(lines) > 0.65
    )

In [1329]:
def unique_keep_order(items):
    seen = set()
    result = []

    for item in items:
        if not item:
            continue

        cleaned = clean_entity_text(str(item))

        key = cleaned.lower()

        if key not in seen:
            result.append(cleaned)
            seen.add(key)

    return result

In [1330]:
def extract_metadata(document_id, text):
    text = normalize_text(text)

    language = get_document_language(document_id)
    nlp = load_spacy_model(language)

    doc = nlp(text[:100000]) if nlp else None

    title = extract_title(text)
    dates = extract_dates(text, doc=doc)
    subtitle = extract_subtitle(text, title=title)
    contributors = extract_contributors(text, doc=doc)
    contact_person = extract_contact_person(text, doc=doc)

    return {
        "document_id": document_id,
        "language": language,
        "title": title,
        "subtitle": subtitle,
        "description": extract_description(text, title=title),
        "contributors": contributors["persons"],
        "contributors_structured": contributors,
        "contributors_confidence": contributors["confidence"],
        "contact_person": contact_person,
        "dates": dates,
        "processed_at": datetime.now().isoformat()
    }

In [1331]:
def save_metadata(document_id, metadata):
    output_path = GOLD_META_FOLDER / f"{document_id}_meta.json"

    output_path.write_text(
        json.dumps(metadata, indent=4, ensure_ascii=False),
        encoding="utf-8"
    )

    print(f"Saved metadata: {output_path}")

In [1332]:
def run_gold_meta_layer():
    silver_files = sorted(SILVER_FOLDER.glob("doc_*_silver.json"))

    if not silver_files:
        print("No silver files found.")
        return

    for silver_file in silver_files:
        document_id = silver_file.stem.replace("_silver", "")

        print(f"Extracting metadata for {document_id}...")

        text = read_metadata_text(document_id)

        if not text:
            print(f"No metadata text found for {document_id}.")
            continue

        metadata = extract_metadata(document_id, text)

        save_metadata(document_id, metadata)

    print("Gold metadata layer completed.")

In [1333]:
run_gold_meta_layer()

Extracting metadata for doc_01...
Saved metadata: ..\..\Data\gold_meta\doc_01_meta.json
Gold metadata layer completed.
